In [6]:
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

project = Path.cwd().parent

features = pd.read_csv(
    project / "data" / "processed" / "customer_features.csv"
)

features["SIGNUP_DATE"] = pd.to_datetime(features["SIGNUP_DATE"])
features = features.sort_values("SIGNUP_DATE").reset_index(drop=True)

cutoff_date = features["SIGNUP_DATE"].quantile(0.8)

train_mask = features["SIGNUP_DATE"] <= cutoff_date
test_mask = features["SIGNUP_DATE"] > cutoff_date

X = features.drop(columns=["CHURNED", "CUSTOMER_ID", "SIGNUP_DATE"])
y = features["CHURNED"].astype(int)

X_train = X.loc[train_mask].copy()
X_test = X.loc[test_mask].copy()
y_train = y.loc[train_mask].copy()
y_test = y.loc[test_mask].copy()

categorical_columns = X_train.select_dtypes(
    include=["object", "string"]
).columns.tolist()

numeric_columns = X_train.select_dtypes(
    exclude=["object", "string"]
).columns.tolist()

preprocessor = ColumnTransformer([
    (
        "num",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        numeric_columns
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_columns
    )
])

logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

random_forest_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

logistic_model.fit(X_train, y_train)
random_forest_model.fit(X_train, y_train)

logistic_prob = logistic_model.predict_proba(X_test)[:, 1]
random_forest_prob = random_forest_model.predict_proba(X_test)[:, 1]

pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Train Customers": [len(X_train), len(X_train)],
    "Validation Customers": [len(X_test), len(X_test)],
    "Predictions": [len(logistic_prob), len(random_forest_prob)]
})

,Model,Train Customers,Validation Customers,Predictions
0,Logistic Regression,1780,445,445
1,Random Forest,1780,445,445


In [7]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)

logistic_pred = (logistic_prob >= 0.5).astype(int)
random_forest_pred = (random_forest_prob >= 0.5).astype(int)

results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, logistic_pred),
        accuracy_score(y_test, random_forest_pred)
    ],
    "Precision": [
        precision_score(y_test, logistic_pred, zero_division=0),
        precision_score(y_test, random_forest_pred, zero_division=0)
    ],
    "Recall": [
        recall_score(y_test, logistic_pred, zero_division=0),
        recall_score(y_test, random_forest_pred, zero_division=0)
    ],
    "F1": [
        f1_score(y_test, logistic_pred, zero_division=0),
        f1_score(y_test, random_forest_pred, zero_division=0)
    ],
    "PR-AUC": [
        average_precision_score(y_test, logistic_prob),
        average_precision_score(y_test, random_forest_prob)
    ]
}).round(4)

results

,Model,Accuracy,Precision,Recall,F1,PR-AUC
0,Logistic Regression,0.8652,0.4217,0.7447,0.5385,0.6204
1,Random Forest,0.7596,0.2581,0.6809,0.3743,0.6619


In [9]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]

threshold_results = []

for threshold in thresholds:
    for model_name, probabilities in [
        ("Logistic Regression", logistic_prob),
        ("Random Forest", random_forest_prob)
    ]:
        predictions = (probabilities >= threshold).astype(int)

        threshold_results.append({
            "Model": model_name,
            "Threshold": threshold,
            "Precision": precision_score(
                y_test, predictions, zero_division=0
            ),
            "Recall": recall_score(
                y_test, predictions, zero_division=0
            ),
            "F1": f1_score(
                y_test, predictions, zero_division=0
            )
        })

threshold_results = pd.DataFrame(threshold_results).round(4)

threshold_results

,Model,Threshold,Precision,Recall,F1
0,Logistic Regression,0.2,0.3008,0.8511,0.4444
1,Random Forest,0.2,0.1557,0.9574,0.2679
2,Logistic Regression,0.3,0.3273,0.7660,0.4586
3,Random Forest,0.3,0.1556,0.8511,0.2632
4,Logistic Regression,0.4,0.3763,0.7447,0.5000
5,Random Forest,0.4,0.1729,0.7872,0.2835
6,Logistic Regression,0.5,0.4217,0.7447,0.5385
7,Random Forest,0.5,0.2581,0.6809,0.3743
8,Logistic Regression,0.6,0.4795,0.7447,0.5833
9,Random Forest,0.6,0.5577,0.6170,0.5859
